# Filtrage EUR / montant, et pourquoi le XOR renvoie parfois des NaN

Ce notebook reprend ton code (filtre EUR, filtre montant, puis XOR), le corrige l�g�rement,
et explique pourquoi un `^` (XOR) entre deux conditions peut produire des `NaN` / `<NA>`
au lieu de `True`/`False`.

In [1]:
import pandas as pd

ctr = pd.read_csv("../data/CTR.csv", sep=";", encoding="cp1252")
ctr = ctr.rename(columns={"SLD_CTR": "argent_dedans"})

# nettoyage minimal : "." -> valeur manquante, puis conversion en nombre
ctr = ctr.replace(".", pd.NA)
ctr["argent_dedans"] = pd.to_numeric(ctr["argent_dedans"], errors="coerce")

ctr.dtypes

IDT_AC             int64
REF_CTR_INN        int64
DAT_OUV_CTR          str
COD_ECV_CTR        int64
DAT_ECV_CTR          str
DAT_CLO_CTR          str
COD_DEV              str
argent_dedans    float64
DAT_MAJ_SLD          str
SLD_DSP              str
MNT_INI              str
dtype: object

## 1. Filtres simples

In [2]:
ctr_EUR = ctr[ctr["COD_DEV"] == "EUR"]
print(ctr_EUR.shape[0], "contrats en EUR")

ctr_1KEUR = ctr[(ctr["COD_DEV"] == "EUR") & (ctr["argent_dedans"] > 1000)]
ctr_1KEUR.head()

193 contrats en EUR


,IDT_AC,REF_CTR_INN,DAT_OUV_CTR,COD_ECV_CTR,DAT_ECV_CTR,DAT_CLO_CTR,COD_DEV,argent_dedans,DAT_MAJ_SLD,SLD_DSP,MNT_INI
18,65500050418,29282017114,2020-12-01,6,2024-08-01,2024-08-01,EUR,1219.68,2025-10-31,NaN,0.00
48,65500115203,90087852236,2009-09-02,4,2009-09-02,NaN,EUR,3718.53,2026-05-18,3718.53,NaN
66,65500153407,29762342182,2025-04-03,4,2025-04-03,NaN,EUR,3279.78,2026-05-15,3279.78,NaN
72,65500163296,29762182482,2021-07-23,4,2021-07-23,NaN,EUR,26168.60,2026-05-27,26168.60,NaN
96,65500204394,90080911729,2011-11-17,4,2018-11-25,NaN,EUR,79156.24,2026-05-27,NaN,NaN


In [3]:
ctr_1k = ctr[ctr["argent_dedans"] > 1000]
ctr_1k.head()

,IDT_AC,REF_CTR_INN,DAT_OUV_CTR,COD_ECV_CTR,DAT_ECV_CTR,DAT_CLO_CTR,COD_DEV,argent_dedans,DAT_MAJ_SLD,SLD_DSP,MNT_INI
18,65500050418,29282017114,2020-12-01,6,2024-08-01,2024-08-01,EUR,1219.68,2025-10-31,NaN,0.00
48,65500115203,90087852236,2009-09-02,4,2009-09-02,NaN,EUR,3718.53,2026-05-18,3718.53,NaN
66,65500153407,29762342182,2025-04-03,4,2025-04-03,NaN,EUR,3279.78,2026-05-15,3279.78,NaN
72,65500163296,29762182482,2021-07-23,4,2021-07-23,NaN,EUR,26168.60,2026-05-27,26168.60,NaN
96,65500204394,90080911729,2011-11-17,4,2018-11-25,NaN,EUR,79156.24,2026-05-27,NaN,NaN


## 2. XOR entre deux conditions

In [4]:
cond_EUR = ctr["COD_DEV"] == "EUR"
cond_1k = ctr["argent_dedans"] > 1000

print("dtype cond_EUR :", cond_EUR.dtype)
print("dtype cond_1k  :", cond_1k.dtype)

ctr_xor = ctr[cond_EUR ^ cond_1k]
ctr_xor[["IDT_AC", "COD_DEV", "argent_dedans"]].head()

dtype cond_EUR : bool
dtype cond_1k  : bool


,IDT_AC,COD_DEV,argent_dedans
0,65500004701,EUR,NaN
1,65500006391,EUR,NaN
2,65500007774,EUR,NaN
3,65500008787,EUR,NaN
4,65500014230,EUR,NaN


## 3. Pourquoi le XOR peut renvoyer des NaN

`^` sur deux `Series` booléennes suit la logique classique **seulement si les deux
côtés sont de vrais booléens numpy** (`dtype: bool`). Dans ce cas il n'y a jamais de
NaN : chaque case est True ou False.

Le problème apparaît quand une des deux conditions vient d'une colonne de type
**nullable** (`string`, `boolean`, `Int64`, `Float64`... avec un I ou B majuscule,
différent du `str`/`float64` normal). Ces types gèrent les valeurs manquantes avec
`pd.NA`, et suivent la **logique à trois valeurs de Kleene** : True / False / *inconnu*.

Dans cette logique :
- `NA == "EUR"` -> `NA` (on ne sait pas, la valeur est manquante)
- `NA ^ True`  -> `NA` (impossible de dire si c'est "un seul des deux" vrai)
- `NA ^ False` -> `NA` (même raison)

Autrement dit : dès qu'une case est manquante d'un côté, le résultat du XOR pour cette
ligne est *indeterminé*, donc `NaN`/`<NA>` au lieu de True/False. `&` et `|` ont le même
comportement (c'est la logique à 3 valeurs standard en SQL et en logique booléenne),
mais on le remarque surtout avec `^` car il est moins utilisé et le résultat surprend plus.

Petite démo qui reproduit le phénomène :

In [5]:
demo = pd.Series(["EUR", pd.NA, "USD"], dtype="string")   # dtype nullable (string, pas str)
cond_a = demo == "EUR"
cond_b = pd.Series([True, False, True])

print(cond_a)          # <NA> à la ligne manquante
print(cond_a ^ cond_b)  # <NA> se propage dans le XOR

0     True
1     <NA>
2    False
dtype: boolean
0    False
1     <NA>
2     True
dtype: boolean


## 4. Comment éviter les NaN dans le XOR

Deux solutions simples :

1. **Décider ce que "manquant" doit valoir** avant le XOR, avec `.fillna(False)` :

In [6]:
ctr_xor_propre = ctr[(cond_EUR.fillna(False) ^ cond_1k.fillna(False))]
ctr_xor_propre[["IDT_AC", "COD_DEV", "argent_dedans"]].head()

,IDT_AC,COD_DEV,argent_dedans
0,65500004701,EUR,NaN
1,65500006391,EUR,NaN
2,65500007774,EUR,NaN
3,65500008787,EUR,NaN
4,65500014230,EUR,NaN


2. **Forcer un vrai booléen numpy** avec `.astype(bool)` (attention : `.astype(bool)`
   transforme aussi `NA` en `True`, donc `.fillna(False)` avant reste la méthode la
   plus sûre et la plus claire).